# 05 - DPV Basics: Pulse Voltammetry

Differential pulse voltammetry (DPV) has its own eCAT object because the useful metadata and analysis are different from CV. This notebook loads paired Ar/CO2 Fe-tpyPY2Me DPV traces, inspects the pulse settings, plots the traces, finds the dominant peak potential, and fits two overlapping peaks with explicit guesses.

## Import eCAT And Set Paths

This uses the same short setup pattern as notebooks 03 and 04. The data folder contains two CH Instruments DPV exports copied from the Fe/PhOH example set: one Ar trace and one CO2 trace.

In [1]:
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent


def display_path(path):
    path = Path(path)
    try:
        return str(path.resolve().relative_to(ROOT.resolve()))
    except ValueError:
        return path.name


import pandas as pd
import ecat as e

DATA_DIR = ROOT / "examples" / "data" / "fe_phoh_dpv"
EXPORT_DIR = ROOT / "notebooks" / "_outputs"
EXPORT_DIR.mkdir(exist_ok=True)

e.plotting_style("notebook")
print("eCAT version:", getattr(e, "__version__", "unknown"))
print("Example data:", display_path(DATA_DIR))
print("Text files:", len(list(DATA_DIR.glob("*.txt"))))

eCAT version: 0.1.0b6
Example data: examples/data/fe_phoh_dpv
Text files: 2


## Load DPV Files

`get_data()` promotes these CH exports to `dpv` objects automatically. Keep `reference mode` set to `"none"` here so the first pass shows the exported potentials exactly as collected.

In [2]:
dpvs = e.get_data({
    "folder path": str(DATA_DIR),
    "software": "CH",
    "reference mode": "none",
    "print": True,
})

Searching recursively through:
 examples/data/fe_phoh_dpv
2 supported text files found.

[Conditions] Exp Type: DPV, Solvent: MeCN, Compounds: 0.1 M TBAPF₆, 3 mM Fc, 1 mM Fe-tpyPY2Me, Scan Window: [-1.2, -0.7], Segments: 1, Amplitude: 10 mV, Pulse Width: 50 ms, Sample Width: 16.7 ms, Pulse Period: 500 ms, IR Comp Percent: 100 %


,Gas
[0],Ar
[1],CO2


## Inspect One DPV Object

A DPV object still has the familiar `info()`, `stats()`, `x()`, `y()`, and `plot()` methods. The DPV-specific stats include pulse amplitude, pulse width, sample width, and pulse period, which are usually more useful than scan rate for this technique.

In [3]:
dpv_ar = e.filter(dpvs, {"gas": "Ar"}, {"print": False})[0]
dpv_co2 = e.filter(dpvs, {"gas": "CO2"}, {"print": False})[0]

e.show(dpv_ar);

,Metric,Value
0,Name,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_DPV_-0.7_to_-1.2V
1,Timestamp,2026-05-07 15:21:34
2,Creation Time,2026-06-10 21:45:43.785275
3,Modification Time,2026-06-10 21:45:43.785662
4,Reference Label,Fc/Fc+
5,IR Comp Resistance,118 ohm
6,IR Uncomp Resistance,0 ohm
7,IR Comp Percent,100 %
8,Solvent,MeCN
9,Gas,Ar


## Plot A Single DPV Trace

The default DPV plot uses potential on the x-axis and current on the y-axis. You can pass the same display options used elsewhere, including title, subtitle, units, and legend settings.

In [4]:
ax = dpv_ar.plot({
    "title": "Ar DPV Trace",
    "subtitle": "Fe-tpyPY2Me with Fc internal reference present",
})

## Compare Ar And CO2

`multiplot()` also works for DPV lists. Here the two traces share the same pulse settings and scan window, so the overlay is a quick visual comparison of the gas condition.

In [5]:
ax = e.multiplot(dpvs, {
    "title": "Ar vs CO$_2$ DPV",
    "subtitle": "Same pulse settings and scan window",
    "labels": ["Ar", "CO$_2$"],
    "legend": True,
})

## Find The Dominant DPV Peak

`peak_potential()` is smart enough to find the dominant DPV feature without options. For final analysis, add a `guess potential` so the notebook records which feature you intended to measure.

In [6]:
peak_rows = []
for obj in dpvs:
    Ep, idx = obj.peak_potential({
        "guess potential": -0.97,
        "plot": False,
        "print": False,
    })
    peak_rows.append({
        "name": obj.name,
        "gas": obj.gas,
        "Ep_V": float(Ep),
        "index": int(idx),
    })

peak_table = pd.DataFrame(peak_rows)
peak_table

,name,gas,Ep_V,index
0,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_DPV_-0...,Ar,-0.973,272
1,MeCN_CO2_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_DPV_-...,CO2,-0.969,268


## Plot Peak Diagnostics

Turn plotting on when you want to see the selected point. `guess potential` guides the selection, and `troubleshoot: True` also marks candidate extrema so you can judge whether the peak-picking settings are reasonable. Keep this call minimal: DPV peak selection options are intentionally narrower than full plot options.

In [7]:
_ = dpv_co2.peak_potential({
    "guess potential": -0.97,
    "troubleshoot": True,
})

Metric,Value
Ep,-969.0 mV


SG window=25, polyorder=3, prominence=2.3293763008146945e-07, prominence mode=guess local


## Fit Overlapping Peaks

The paired Fe/Fc region is close enough that `fit_overlapping_peaks()` needs two explicit guesses. The guesses are approximate peak centers in volts on the active x-axis. Start with a simple fit, then tighten options such as `fit window`, `center window`, `sigma guess`, and labels if the chemistry requires it.

In [8]:
fit_result = dpv_co2.fit_overlapping_peaks({
    "guess potentials": [-0.94, -1.02],
    "title": "CO$_2$ DPV Overlapping Peak Fit",
    "data label": "CO$_2$ DPV",
})

fit_result

component  potential (V)  amplitude (A)  sigma (V)  baseline current (A)  current (A)
   peak 1      -0.905905      -0.000008   0.036161         -5.616589e-07    -0.000011
   peak 2      -0.981371      -0.000011   0.042685         -5.393121e-07    -0.000012


,component,potential (V),amplitude (A),sigma (V),baseline current (A),current (A)
0,peak 1,-0.905905,-0.000008,0.036161,-5.616589e-07,-0.000011
1,peak 2,-0.981371,-0.000011,0.042685,-5.393121e-07,-0.000012


## Save A DPV Figure

Use ordinary Matplotlib saving on the returned axes. Keep exports in `notebooks/_outputs` so generated files stay separate from raw example data.

In [9]:
ax = e.multiplot(dpvs, {
    "print": False,
    "title": "Ar vs CO$_2$ DPV",
    "labels": ["Ar", "CO$_2$"],
})

# ax.figure.savefig(EXPORT_DIR / "dpv_ar_co2_overlay.png", dpi=300, bbox_inches="tight")